# TPV Ultra Smart v7.0 — Compilar APK v3

## Instrucciones:
1. Ejecuta las celdas en orden (Shift+Enter o el boton Play)
2. La APK se guarda en tu Google Drive al terminar
3. Descargala e instalala en tu Android

**Tiempo estimado: 20-40 minutos la primera vez**

---
### Correcciones v3 (todas las mejoras de la sesion):
- Kivy NO se instala via pip
- Buildozer responde automaticamente al aviso de root
- Cython fijado a 0.29.36
- source.dir añadido automaticamente
- p4a.branch = master (estable)
- Detecta proyecto en MyDrive o subcarpeta TPV_APK
- Aplana estructura de carpetas para compilacion (backend/, frontend/, etc.)
- Copia iconos al directorio raiz para buildozer
- Añade icono a buildozer.spec automaticamente
- Compilacion con os.system (output en tiempo real)

## CELDA 1 — Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado')

## CELDA 2 — Detectar ruta del proyecto

In [ ]:
import os

RUTAS_BUSCAR = [
    '/content/drive/MyDrive/TPV_APK',
    '/content/drive/MyDrive/TPV_Supabase',
    '/content/drive/MyDrive/TPV',
    '/content/drive/MyDrive',
]

RUTA_PROYECTO = None
for ruta in RUTAS_BUSCAR:
    if os.path.exists(ruta) and os.path.exists(os.path.join(ruta, 'main.py')):
        RUTA_PROYECTO = ruta
        break

if RUTA_PROYECTO:
    print('Proyecto encontrado en: ' + RUTA_PROYECTO)
    print('Contenido:')
    for item in sorted(os.listdir(RUTA_PROYECTO)):
        print('  ' + item)
else:
    print('No se encontro main.py. Contenido de MyDrive:')
    for item in sorted(os.listdir('/content/drive/MyDrive')):
        print('  ' + item)
    RUTA_PROYECTO = '/content/drive/MyDrive'
    print('Usando raiz de MyDrive: ' + RUTA_PROYECTO)

## CELDA 3 — Instalar dependencias del sistema
Tarda 3-5 minutos

In [ ]:
import os
print('Instalando dependencias del sistema...')
os.system('apt-get update -qq')
os.system('apt-get install -y -qq git zip unzip wget openjdk-17-jdk python3-pip autoconf automake libtool pkg-config zlib1g-dev libncurses5-dev libffi-dev libssl-dev build-essential ccache libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev libportmidi-dev libswscale-dev libavformat-dev libavcodec-dev libgstreamer1.0-dev gstreamer1.0-plugins-base libgl1-mesa-dev libgles2-mesa-dev')
import subprocess
java = subprocess.run(['java', '-version'], capture_output=True, text=True)
print('Java: ' + java.stderr.split('\n')[0])
print('Dependencias listas')

## CELDA 4 — Instalar Buildozer y Cython

In [ ]:
import os
print('Instalando Cython 0.29.36...')
os.system('pip install -q cython==0.29.36')
print('Instalando Buildozer 1.5.0...')
os.system('pip install -q buildozer==1.5.0')
print('Instalando dependencias auxiliares...')
os.system('pip install -q colorama appdirs sh jinja2 "virtualenv<20.22.0"')
import subprocess
r = subprocess.run('pip show buildozer | grep Version', shell=True, capture_output=True, text=True)
print('Buildozer: ' + r.stdout.strip())
print('Instalacion completada')

## CELDA 5 — Copiar y aplanar proyecto para compilacion
Si tu proyecto tiene carpetas (backend/, frontend/, etc.) las aplana automaticamente para que Buildozer encuentre todos los archivos.

In [ ]:
import shutil, os

DIR_TRABAJO = '/content/TPV_build'

# Limpiar compilacion anterior
if os.path.exists(DIR_TRABAJO):
    shutil.rmtree(DIR_TRABAJO)
    print('Directorio anterior limpiado')

os.makedirs(DIR_TRABAJO)

# Archivos que van en la raiz del build
ARCHIVOS_RAIZ = [
    'main.py', 'buildozer.spec', 'requirements.txt',
    'app.py', 'database.py', 'tienda_routes.py',
    'supabase_sync.py', 'pwa_routes.py', 'tpv_rutas.py',
    'index.html',
    'tpv_main.js', 'tpv_auth.js', 'tpv_tienda.js',
    'tpv_export.js', 'tpv_debugger.js', 'service-worker.js',
    'manifest.json',
    'pwa-icon-192.png', 'pwa-icon-512.png',
    'icon-144.png', 'icon-96.png', 'icon-72.png', 'icon-48.png',
]

# Subcarpetas donde buscar archivos si el proyecto esta organizado
SUBCARPETAS_BUSCAR = [
    '',                          # raiz
    'backend',
    'frontend/templates',
    'frontend/static/js',
    'frontend/static/icons',
    'frontend/static',
    'static/js',
    'static',
    'templates',
]

copiados = []
no_encontrados = []

for archivo in ARCHIVOS_RAIZ:
    encontrado = False
    for sub in SUBCARPETAS_BUSCAR:
        ruta = os.path.join(RUTA_PROYECTO, sub, archivo) if sub else os.path.join(RUTA_PROYECTO, archivo)
        if os.path.exists(ruta):
            destino = os.path.join(DIR_TRABAJO, archivo)
            shutil.copy2(ruta, destino)
            copiados.append(archivo)
            encontrado = True
            break
    if not encontrado:
        no_encontrados.append(archivo)

# Copiar carpeta uploads si existe (imagenes de productos)
for sub_uploads in ['frontend/static/uploads', 'static/uploads', 'uploads']:
    ruta_uploads = os.path.join(RUTA_PROYECTO, sub_uploads)
    if os.path.exists(ruta_uploads):
        shutil.copytree(ruta_uploads, os.path.join(DIR_TRABAJO, 'uploads'))
        print('Carpeta uploads copiada')
        break

print('Archivos copiados al build:')
for f in sorted(os.listdir(DIR_TRABAJO)):
    tam = os.path.getsize(os.path.join(DIR_TRABAJO, f))
    print('  [OK] ' + f + ' (' + str(tam) + ' bytes)')

if no_encontrados:
    opcionales = ['pwa_routes.py', 'tpv_rutas.py', 'service-worker.js',
                  'manifest.json', 'icon-144.png', 'icon-96.png',
                  'icon-72.png', 'icon-48.png', 'pwa-icon-192.png', 'pwa-icon-512.png']
    criticos = [f for f in no_encontrados if f not in opcionales]
    if criticos:
        print('ERROR: Faltan archivos criticos: ' + str(criticos))
        raise FileNotFoundError('Faltan: ' + str(criticos))
    else:
        print('Archivos opcionales no encontrados (no es error): ' + str(no_encontrados))

os.chdir(DIR_TRABAJO)
print('Directorio actual: ' + os.getcwd())

## CELDA 6 — Corregir buildozer.spec automaticamente

In [ ]:
import re, os

os.chdir(DIR_TRABAJO)

with open('buildozer.spec', 'r') as f:
    spec = f.read()

# Arreglo 1: source.dir
if 'source.dir' not in spec:
    spec = spec.replace('source.main = main.py', 'source.dir = .\nsource.main = main.py')
    print('Aplicado: source.dir = .')
else:
    print('OK: source.dir ya existe')

# Arreglo 2: p4a.branch = master
if 'p4a.branch = develop' in spec:
    spec = spec.replace('p4a.branch = develop', 'p4a.branch = master')
    print('Aplicado: p4a.branch = master')
elif 'p4a.branch' not in spec:
    spec += '\np4a.branch = master\n'
    print('Aplicado: p4a.branch = master (anadido)')
else:
    print('OK: p4a.branch ya configurado')

# Arreglo 3: icono si existe en el build
icono_path = os.path.join(DIR_TRABAJO, 'pwa-icon-512.png')
if os.path.exists(icono_path):
    if 'icon.filename' not in spec:
        spec = spec.replace(
            '# icon.filename',
            'icon.filename = %(source.dir)s/pwa-icon-512.png\n# icon.filename'
        )
        if 'icon.filename = ' not in spec:
            spec += '\nicon.filename = %(source.dir)s/pwa-icon-512.png\n'
        print('Aplicado: icono pwa-icon-512.png')
    else:
        print('OK: icono ya configurado')
else:
    print('Icono no encontrado, se usara el icono por defecto')

# Arreglo 4: include_exts para incluir imagenes y json
if 'source.include_exts' in spec:
    m = re.search(r'^source\.include_exts\s*=\s*(.+)$', spec, re.MULTILINE)
    if m:
        exts_actuales = m.group(1).strip()
        exts_nuevas = exts_actuales
        for ext in ['json', 'png', 'jpg', 'html', 'js', 'css']:
            if ext not in exts_nuevas:
                exts_nuevas += ',' + ext
        spec = spec.replace('source.include_exts = ' + exts_actuales,
                            'source.include_exts = ' + exts_nuevas)
        print('Aplicado: source.include_exts actualizado')

with open('buildozer.spec', 'w') as f:
    f.write(spec)

# Mostrar configuracion final
print('\nConfiguracion buildozer.spec:')
claves = ['title', 'package.name', 'version', 'requirements',
          'android.api', 'android.minapi', 'android.archs',
          'p4a.branch', 'source.dir', 'icon.filename']
for key in claves:
    m = re.search(r'^' + re.escape(key) + r'\s*=\s*(.+)$', spec, re.MULTILINE)
    if m:
        print('  ' + key + ' = ' + m.group(1).strip())

print('\nbuildozer.spec listo')

## CELDA 7 — COMPILAR LA APK
**Esta celda tarda 20-40 minutos la primera vez.**
El output aparece en tiempo real. No cierres la pestana.

In [ ]:
import os, glob

os.chdir(DIR_TRABAJO)
print('Compilando APK... (20-40 minutos, no cierres la pestana)')
print('-' * 60)

os.system('yes | buildozer -v android debug 2>&1')

print('-' * 60)

apks = glob.glob(DIR_TRABAJO + '/bin/*.apk')
if apks:
    tam = os.path.getsize(apks[0]) / (1024*1024)
    print('APK generada: ' + os.path.basename(apks[0]))
    print('Tamano: ' + str(round(tam, 1)) + ' MB')
else:
    print('ERROR: APK no generada. Revisa el output de arriba.')

## CELDA 8 — Guardar APK en Google Drive

In [ ]:
import glob, shutil, os
from datetime import datetime

apks = glob.glob(DIR_TRABAJO + '/bin/*.apk')

if apks:
    nombre_apk = 'TPV_UltraSmart_' + datetime.now().strftime('%Y%m%d_%H%M') + '.apk'
    destino = '/content/drive/MyDrive/' + nombre_apk
    shutil.copy2(apks[0], destino)
    tam = os.path.getsize(destino) / (1024*1024)
    print('APK guardada en Google Drive:')
    print('  Nombre: ' + nombre_apk)
    print('  Tamano: ' + str(round(tam, 1)) + ' MB')
    print('  Ruta: ' + destino)
    print('')
    print('Para instalar en Android:')
    print('  1. Descarga el .apk desde Google Drive al movil')
    print('  2. Ajustes -> Seguridad -> Fuentes desconocidas (activar)')
    print('  3. Toca el archivo .apk para instalar')
else:
    print('No hay APK. Ejecuta primero la Celda 7.')

## CELDA 9 — Descargar APK directamente al navegador (alternativa)

In [ ]:
from google.colab import files
import glob

apks = glob.glob(DIR_TRABAJO + '/bin/*.apk')
if apks:
    print('Descargando: ' + os.path.basename(apks[0]))
    files.download(apks[0])
else:
    print('No hay APK. Ejecuta primero la Celda 7.')